# Mainnet WETH/USDC

This notebook now uses the shared `second_level_markets.py` pipeline. It either builds a real one-week, second-by-second reserve reconstruction on compatible on-chain markets, or writes an explicit blocker manifest when that is not honest to do.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from second_level_markets import SECOND_LEVEL_MARKETS, dataset_summary_frame, load_or_build_second_level_dataset

MARKET_KEY = '08_mainnet_weth_usdc'
MARKET_SPEC = SECOND_LEVEL_MARKETS[MARKET_KEY]
REFRESH_SECOND_LEVEL_DATA = False

market_dataset = load_or_build_second_level_dataset(MARKET_KEY, project_root=PROJECT_ROOT, refresh=REFRESH_SECOND_LEVEL_DATA)

summary_rows = [
    {'field': 'chain_name', 'value': MARKET_SPEC.chain_name},
    {'field': 'pair_label', 'value': MARKET_SPEC.pair_label},
    {'field': 'sample_window_days', 'value': MARKET_SPEC.sample_window_days},
    {'field': 'state_frequency', 'value': '1s'},
    {'field': 'supported', 'value': MARKET_SPEC.supported},
    {'field': 'primary_trade_size_base', 'value': MARKET_SPEC.primary_trade_size_base},
    {'field': 'trade_sizes_base', 'value': ', '.join(str(x) for x in MARKET_SPEC.trade_sizes_base)},
    {'field': 'notes', 'value': ' | '.join(MARKET_SPEC.notes)},
]

display(pd.DataFrame(summary_rows))
display(dataset_summary_frame(market_dataset))

if market_dataset['status'] == 'blocked':
    display(pd.DataFrame({'blocker': market_dataset['blockers']}))
else:
    display(market_dataset['pair_metadata'])
    display(market_dataset['size_sensitivity'])
    display(pd.DataFrame(market_dataset['qc_report'].items(), columns=['check', 'value']))


AttributeError: Can only use .dt accessor with datetimelike values

In [ ]:
if market_dataset["status"] == "blocked":
    display(Markdown("""## Blocked

This market is intentionally left without synthetic second-level output. The blocker list above explains what additional infrastructure or venue coverage would be required."""))
else:
    arb_labels = market_dataset["arb_labels"]
    opportunity_windows = market_dataset["opportunity_windows"]
    key_columns = [
        "timestamp",
        "buy_dex",
        "sell_dex",
        "trade_size_base",
        "gross_edge_bps",
        "fee_cost_bps",
        "gas_cost_quote",
        "net_edge_quote",
        "net_edge_bps",
        "opportunity_flag",
    ]
    best_seconds = arb_labels.sort_values("net_edge_bps", ascending=False).head(20)
    display(best_seconds.loc[:, [column for column in key_columns if column in best_seconds.columns]])
    display(opportunity_windows.head(20))


In [ ]:
if market_dataset['status'] == 'blocked':
    display(Markdown('No second-level plot is generated for blocked markets.'))
else:
    pool_state = market_dataset['pool_state'].copy()
    arb_labels = market_dataset['arb_labels'].copy()
    opportunity_windows = market_dataset['opportunity_windows'].copy()

    display(Markdown('## Visual Checks'))
    fig, axes = plt.subplots(2, 1, figsize=(16, 10), constrained_layout=True)

    price_plot = (
        pool_state.assign(plot_timestamp=pool_state['timestamp'].dt.floor('5min'))
        .groupby(['plot_timestamp', 'dex'], as_index=False)
        .agg(mid_price_quote_per_base=('mid_price_quote_per_base', 'last'))
    )
    for dex, group in price_plot.groupby('dex'):
        axes[0].plot(group['plot_timestamp'], group['mid_price_quote_per_base'], label=dex, linewidth=1.5)
    axes[0].set_title('5-minute sampled mid-price by venue')
    axes[0].set_ylabel('Quote per base')
    axes[0].legend()

    if not opportunity_windows.empty:
        best_window = opportunity_windows.iloc[0]
        mask = (
            (arb_labels['timestamp'] >= best_window['start_timestamp'])
            & (arb_labels['timestamp'] <= best_window['end_timestamp'])
        )
        plot_scores = arb_labels.loc[mask].sort_values('timestamp')
        axes[1].set_title('Top positive window net edge')
    else:
        plot_scores = arb_labels.nlargest(min(400, len(arb_labels)), 'net_edge_bps').sort_values('timestamp')
        axes[1].set_title('Top scored seconds by net edge')

    axes[1].plot(plot_scores['timestamp'], plot_scores['net_edge_bps'], linewidth=1.5)
    axes[1].axhline(0.0, color='black', linestyle='--', linewidth=1)
    axes[1].set_ylabel('Net edge (bps)')
    axes[1].set_xlabel('Timestamp (UTC)')
    plt.show()


In [ ]:
import json

manifest_path = PROJECT_ROOT / 'outputs' / MARKET_SPEC.slug / 'metadata' / 'dataset_manifest.json'
display(Markdown(f'## Manifest\n\nSaved manifest: `{manifest_path}`'))
if manifest_path.exists():
    manifest_payload = json.loads(manifest_path.read_text())
    display(pd.json_normalize(manifest_payload, sep='.', max_level=1).T.reset_index().rename(columns={'index': 'field', 0: 'value'}))
